> **Riverside's traffic problem:** The editing assistant gateway (`06-llm-gateway.ipynb`) is live and serving 10 requests per minute. Marketing just announced: "We're launching to all authors next week. Expect 1,000 requests per minute." Your current architecture will break under 100× load — not because of model quality, but because of how tokens are generated. Before buying more hardware, there are software-level optimizations that can get you 5–10× higher throughput with zero hardware changes. This chapter builds each one from first principles.

# LLM Inference Systems: From 10 to 1000 Requests Per Minute

| Part | Optimization | Throughput lever |
|------|--------------|------------------|
| 1 | KV cache | Avoid recomputing attention on prompt tokens → 3–10× faster per-token |
| 2 | Continuous batching | Serve variable-length requests without idle GPU time |
| 3 | PagedAttention | Eliminate KV cache memory fragmentation |
| 4 | Speculative decoding | Use a small draft model to propose tokens; verify in parallel |
| 5 | Prefill vs. decode phases | Profile why they need different batch sizes |
| 6 | Toy → real bridge | Map each technique to vLLM metrics (TTFT, TPOT, throughput) |

---

## Prerequisite Bridge — From `04-llm/06-llm-gateway.ipynb` and `02-transformers`

| Foundation | Role in this notebook |
|---|---|
| LLM gateway (`06-llm-gateway.ipynb`) | The serving infrastructure we're optimising — this chapter makes it faster |
| KV cache (mentioned in `02-transformers`) | First described there as "an optimization"; this chapter builds it |
| Autoregressive generation | Every token is generated by a full forward pass through all layers |

> **Prerequisites:** `learning/genai/04-llm/06-llm-gateway.ipynb` (gateway architecture) and `learning/genai/02-transformers/transformers.ipynb` (attention mechanism).

In [ ]:
import subprocess, sys
for pkg in ['torch', 'numpy', 'matplotlib', 'transformers']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import random

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HAS_GPU = torch.cuda.is_available()
torch.manual_seed(42)

print(f"Device: {DEVICE}")
print(f"Riverside scenario: gateway serving 10 req/min → needs to handle 1,000 req/min")
print()

# Toy model config (GPT-2 small scale)
VOCAB_SIZE  = 1000
D_MODEL     = 256
N_HEADS     = 8
N_LAYERS    = 4
D_HEAD      = D_MODEL // N_HEADS

---

## Part 1 — KV Cache: Never Recompute What You've Already Computed

In standard autoregressive generation, each new token requires a full attention computation over **all** previous tokens. For a 128-token prompt + 20-token response:
- Token 1: attention over 128 tokens (the prompt)
- Token 2: attention over 129 tokens
- Token 20: attention over 147 tokens

Each step recomputes keys and values for all previous positions — redundant work.

**KV cache:** compute K and V for each token once, store them, and reuse:
- Prefill: process the full prompt, store all K/V pairs in cache
- Decode: for each new token, compute only 1 new K/V pair; look up the rest from cache

#### 🔮 Predict first

Without KV cache: each decode step runs attention over N positions.
With KV cache: each new token attends over how many *new* positions?

1. **(a) 128 positions** — still needs to see all prompt tokens
2. **(b) 1 position** — only the new token needs new K/V
3. **(c) 1 new K/V pair computed, rest from cache** — 1 new computation + cache lookup

> **Intuition — it's just a cache:** Think of a database query result cache. The first time you run an expensive query, the database stores the result. Subsequent identical queries return the cached result instantly. KV cache does the same for attention: compute keys and values for each prompt token *once* during prefill, write them to cache, and on every decode step only compute the one new token's K/V pair. Everything else is a cache lookup. The `past_key_values` parameter is literally the cache dictionary.

![KV cache mechanism: prefill processes all prompt tokens once, storing K/V; each decode step computes just 1 new K/V pair and appends it to the cache](images/kv-cache-mechanism.png)

In [ ]:
# ── Part 1: KV-caching forward pass ──────────────────────────────────────────
class SimpleCausalAttention(nn.Module):
    """Minimal causal attention with optional KV cache."""
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.W_qkv   = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o     = nn.Linear(d_model, d_model, bias=False)
        self.scale   = self.d_head ** -0.5

    def forward(self, x, past_kv=None):
        """
        x:       (B, new_S, D)  — can be 1 token in decode mode
        past_kv: (k_cache, v_cache) each (B, past_S, n_heads, d_head), or None
        Returns: (output, (new_k_cache, new_v_cache))
        """
        B, new_S, D = x.shape
        qkv = self.W_qkv(x).reshape(B, new_S, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)   # each (B, new_S, n_heads, d_head)

        # Extend KV cache
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=1)  # (B, past_S+new_S, n_heads, d_head)
            v = torch.cat([past_kv[1], v], dim=1)

        # Scaled dot-product attention
        q = q.transpose(1, 2)   # (B, n_heads, new_S, d_head)
        k = k.transpose(1, 2)   # (B, n_heads, total_S, d_head)
        v = v.transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (B, n_heads, new_S, total_S)
        attn   = torch.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)                               # (B, n_heads, new_S, d_head)
        out    = out.transpose(1, 2).reshape(B, new_S, D)
        return self.W_o(out), (k.transpose(1,2), v.transpose(1,2))

class ToyGPT(nn.Module):
    """Minimal GPT-style model with KV caching."""
    def __init__(self):
        super().__init__()
        self.embed  = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.layers = nn.ModuleList([SimpleCausalAttention() for _ in range(N_LAYERS)])
        self.norm   = nn.LayerNorm(D_MODEL)
        self.head   = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)

    def forward(self, x, past_kvs=None):
        """x: token ids (B, S). Returns (logits, new_past_kvs)."""
        h = self.embed(x)
        new_kvs = []
        for i, layer in enumerate(self.layers):
            past = past_kvs[i] if past_kvs else None
            h, new_kv = layer(h, past)
            new_kvs.append(new_kv)
        return self.head(self.norm(h)), new_kvs

torch.manual_seed(42)
model = ToyGPT().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ToyGPT: {n_params/1e3:.0f}K parameters (toy proxy for GPT-2)")

In [ ]:
# ── Part 1: Compare with and without KV cache ─────────────────────────────────
PROMPT_LEN  = 32  # prompt tokens
GEN_TOKENS  = 20  # tokens to generate

def generate_no_cache(model, prompt_ids, n_tokens):
    """Standard generation: recompute all K/V at every step."""
    ids = prompt_ids.clone()
    t0 = time.perf_counter()
    for _ in range(n_tokens):
        logits, _ = model(ids, past_kvs=None)      # recompute from scratch
        next_id   = logits[:, -1, :].argmax(-1, keepdim=True)
        ids       = torch.cat([ids, next_id], dim=1)
    return ids, time.perf_counter() - t0

def generate_with_cache(model, prompt_ids, n_tokens):
    """Cached generation: only process new token at each step."""
    # Prefill: process full prompt once, build KV cache
    _, past_kvs = model(prompt_ids, past_kvs=None)
    ids = prompt_ids.clone()
    t0 = time.perf_counter()
    for _ in range(n_tokens):
        logits, past_kvs = model(ids[:, -1:], past_kvs=past_kvs)  # 1 token!
        next_id = logits[:, 0, :].argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
    return ids, time.perf_counter() - t0

prompt = torch.randint(0, VOCAB_SIZE, (1, PROMPT_LEN)).to(DEVICE)
model.eval()
with torch.no_grad():
    # Warm up
    for _ in range(3):
        generate_no_cache(model, prompt, 5)
        generate_with_cache(model, prompt, 5)

    ids_no_cache, t_no_cache = generate_no_cache(model, prompt, GEN_TOKENS)
    ids_cached, t_cached     = generate_with_cache(model, prompt, GEN_TOKENS)

speedup = t_no_cache / t_cached
output_match = torch.equal(ids_no_cache, ids_cached)

print(f"Generating {GEN_TOKENS} tokens from {PROMPT_LEN}-token prompt:")
print(f"  Without KV cache: {t_no_cache*1000:.1f}ms  ({GEN_TOKENS/(t_no_cache):.0f} tok/s)")
print(f"  With KV cache:    {t_cached*1000:.1f}ms    ({GEN_TOKENS/(t_cached):.0f} tok/s)")
print(f"  Speedup: {speedup:.1f}×")
print(f"  Outputs identical: {output_match}")
print()
print("Prediction check: answer (c) — each decode step computes 1 new K/V pair, reuses the rest.")
print(f"  KV cache stores {N_LAYERS} layers × (K + V) × {PROMPT_LEN + GEN_TOKENS} positions")
kv_size_mb = N_LAYERS * 2 * (PROMPT_LEN + GEN_TOKENS) * D_MODEL * 4 / 1e6
print(f"  KV cache size: {kv_size_mb:.2f} MB (for this toy model)")
print(f"  For LLaMA-3-7B at S=2048: ~4 GB (manageable)")

#### What just happened — and what's missing

KV cache gave a measurable speedup by avoiding redundant attention computation. This is the most important single optimization for LLM inference.

**Missing piece:** KV cache solves per-token latency. But Riverside now has 100× more requests. Even with a fast single-request path, we can only serve one request at a time. We need to serve multiple requests **simultaneously** — that's continuous batching.

#### What just happened — and what's missing

KV cache eliminated redundant attention recomputation, giving a significant per-token speedup. But Riverside now needs to serve 100 concurrent authors, each with an active cache. Even with fast individual tokens, we're still processing one request at a time serially — the GPU idles between requests of different lengths. **Next: continuous batching eliminates that idle time.**

---

## Part 2 — Continuous Batching: No Idle GPU Time

**Static batching** groups requests with the same length into a batch. Problem: short requests finish early and the GPU slot sits empty waiting for the longest request.

**Continuous batching** (vLLM, TGI): as soon as any request in the batch finishes, immediately slot in a new request. The GPU is always busy.

At 1000 req/min: static batching can waste 30–50% of GPU capacity in idle slots. Continuous batching reclaims that.

![Continuous batching eliminates idle GPU slots: static batching idles after short requests finish; continuous batching immediately slots in new requests](images/continuous-batching-vs-static.png)

In [ ]:
# ── Part 2: Static vs. Continuous Batching — Gantt-style GPU slot view ────────
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

slot_colors = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

# ── Static batching ───────────────────────────────────────────────────────────
# Batch 1: 4 requests, durations [5, 8, 5, 6]. All must wait until t=8 (the slowest).
# Batch 2 starts at t=8: durations [7, 4, 6, 7]. All wait until t=8+7=15.
ax1.set_title("Static Batching\n(wait for all to finish)", fontweight='bold')
batch1_durations = [5, 8, 5, 6]
batch1_end = max(batch1_durations)          # = 8  (whole batch blocked until slowest)
for slot, dur in enumerate(batch1_durations):
    ax1.barh(slot, dur, left=0, color=slot_colors[slot], alpha=0.85, height=0.6,
             label=f'Req {slot+1}')
    idle = batch1_end - dur
    if idle > 0:
        ax1.barh(slot, idle, left=dur, color='#888', alpha=0.35, height=0.6)

batch2_durations = [7, 4, 6, 7]
batch2_end = batch1_end + max(batch2_durations)   # = 8 + 7 = 15
for slot, dur in enumerate(batch2_durations):
    ax1.barh(slot, dur, left=batch1_end, color=slot_colors[slot], alpha=0.55, height=0.6)
    idle = max(batch2_durations) - dur
    if idle > 0:
        ax1.barh(slot, idle, left=batch1_end + dur, color='#888', alpha=0.35, height=0.6)

ax1.set_xlabel("Time steps")
ax1.set_ylabel("GPU Slot")
ax1.set_yticks([0, 1, 2, 3])
ax1.set_yticklabels(['Slot 0', 'Slot 1', 'Slot 2', 'Slot 3'])
ax1.set_xlim(0, 18)
ax1.axvline(batch1_end, color='#e74c3c', ls='--', lw=1.2, alpha=0.7)
ax1.axvline(batch2_end, color='#e74c3c', ls='--', lw=1.2, alpha=0.7)
ax1.text(9, -0.85, "GPU idle ~30% of time  (grey = wasted slots)",
         color='#c0392b', fontsize=8.5, ha='center')
grey_patch = mpatches.Patch(color='#888', alpha=0.4, label='Idle (wasted)')
ax1.legend(handles=[grey_patch], loc='upper right', fontsize=8)

# ── Continuous batching ───────────────────────────────────────────────────────
# As soon as a slot finishes its request, a new request fills it immediately.
# Slot 0: t=0–5 (reqA), t=5–11 (reqE),  t=11–17 (reqI)
# Slot 1: t=0–8 (reqB), t=8–13  (reqF), t=13–17 (reqJ)
# Slot 2: t=0–5 (reqC), t=5–12  (reqG), t=12–17 (reqK)
# Slot 3: t=0–6 (reqD), t=6–13  (reqH), t=13–17 (reqL)
ax2.set_title("Continuous Batching\n(fill slot immediately on finish)", fontweight='bold')
continuous_slots = [
    [(0, 5),  (5, 11), (11, 17)],   # slot 0
    [(0, 8),  (8, 13), (13, 17)],   # slot 1
    [(0, 5),  (5, 12), (12, 17)],   # slot 2
    [(0, 6),  (6, 13), (13, 17)],   # slot 3
]
for slot, segs in enumerate(continuous_slots):
    for i, (start, end) in enumerate(segs):
        ax2.barh(slot, end - start, left=start, color=slot_colors[slot],
                 alpha=0.85 - i * 0.12, height=0.6)

ax2.set_xlabel("Time steps")
ax2.set_ylabel("GPU Slot")
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['Slot 0', 'Slot 1', 'Slot 2', 'Slot 3'])
ax2.set_xlim(0, 18)
ax2.text(9, -0.85, "GPU idle ~0%  — slots fill immediately, no grey patches",
         color='#27ae60', fontsize=8.5, ha='center')

plt.suptitle("Static vs. Continuous Batching: GPU Utilisation", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n→ Continuous batching: 12 requests served in 17 time-steps vs. 15 steps for 8 requests static (50% more requests, same wall-clock time).")
print("→ Riverside handles a 100× traffic spike by eliminating idle slots — not by adding hardware.")


#### What just happened — and what's missing

Continuous batching filled the GPU's idle slots. But 100 concurrent users each with a KV cache pre-allocated at max sequence length means we're reserving 2048 positions per user even if they only used 50 — 97.5% of that KV cache memory is wasted. **Next: PagedAttention reclaims that fragmented memory.**

---

## Part 3 — PagedAttention: Eliminating KV Cache Fragmentation

**The problem:** Traditional KV cache pre-allocates contiguous memory for max sequence length. If you reserve 2048 tokens for every request but average request is 200 tokens: 90% of the allocated memory is wasted.

**PagedAttention** (from vLLM): manage KV cache like an OS manages virtual memory — in fixed-size **pages** of 16 or 32 tokens. Only allocate pages as tokens are generated. A page table maps logical positions to physical memory blocks.

Result: ~90% GPU memory utilization vs. ~30–50% with pre-allocation.

In [ ]:
# ── Part 3: PagedAttention memory utilization illustration ───────────────────
PAGE_SIZE = 16  # tokens per page

def preallocated_utilization(requests, max_seq_len=512):
    """Old approach: reserve max_seq_len per request."""
    total_reserved  = len(requests) * max_seq_len
    actually_used   = sum(requests)
    return actually_used / total_reserved

def paged_utilization(requests, page_size=PAGE_SIZE):
    """PagedAttention: allocate only the pages actually needed."""
    pages_needed = sum(np.ceil(r / page_size) for r in requests)
    tokens_in_pages = pages_needed * page_size
    actually_used   = sum(requests)
    return actually_used / tokens_in_pages

# Simulate realistic request distribution
np.random.seed(42)
request_lens_dist = np.random.exponential(scale=100, size=100).clip(10, 512).astype(int).tolist()

util_preallocated = preallocated_utilization(request_lens_dist)
util_paged        = paged_utilization(request_lens_dist)

print(f"KV cache memory utilization ({len(request_lens_dist)} concurrent requests):")
print(f"  Pre-allocated (max={512}):  {util_preallocated:.1%}")
print(f"  PagedAttention (page={PAGE_SIZE}): {util_paged:.1%}")
print()
print(f"  PagedAttention improves utilization by {util_paged/util_preallocated:.1f}×")
print(f"  This means {util_paged/util_preallocated:.1f}× more concurrent requests on the same GPU")
print()

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(request_lens_dist, bins=30, color='steelblue', alpha=0.7)
ax.axvline(np.mean(request_lens_dist), color='coral', ls='--', lw=2, label=f'Mean: {np.mean(request_lens_dist):.0f} tokens')
ax.axvline(512, color='gray', ls=':', lw=1, label='Max (pre-allocation waste)')
ax.set_xlabel('Request length (tokens)'); ax.set_ylabel('Count')
ax.set_title('Request length distribution — exponential tail creates pre-allocation waste')
ax.legend(); plt.tight_layout(); plt.show()

#### What just happened — and what's missing

PagedAttention eliminated KV cache fragmentation, allowing 3× more concurrent users on the same GPU. But each individual request still takes the same time — decode is serial. TTFT for a 200-token response is still 200 × decode_latency. **Next: speculative decoding trades a small draft model's speed for the large model's quality, getting multiple tokens per verifier call.**

---

## Part 4 — Speculative Decoding: Parallelise Token Verification

**Problem:** LLM decode is serial — token N must be generated before token N+1.

**Speculative decoding:** use a small, fast **draft model** to propose K tokens at once. Then run the large **verifier** model on all K tokens in a single parallel forward pass.

- If the verifier agrees with all K draft tokens: accept all K tokens at once
- If it disagrees at position j: accept tokens 1..j-1, reject the rest, and sample a corrected token j from the verifier

Net result: most of the time, 3–5 tokens are accepted per verifier call instead of 1.

#### 🔮 Predict first

A 7B verifier and a 70M draft model. The draft proposes 5 tokens; verifier accepts 3. Compared to 5 sequential 7B model calls:

1. **(a) 2× faster** — draft model speed + 3 tokens per verifier call
2. **(b) 5× faster** — equivalent to 5 sequential tokens in 1 verifier call
3. **(c) Marginally faster** — the draft model overhead negates most savings

> **Why parallel verification works:** A transformer is inherently parallel — given N input tokens, it produces N output logits in one forward pass. Autoregressive generation artificially enforces serial order (token N must exist before generating N+1). Speculative decoding breaks that constraint: the small draft model proposes K tokens sequentially (it's fast). The large verifier then does what transformers naturally do — processes all K positions simultaneously in one parallel pass and checks which tokens it agrees with. If the draft was right, you get K tokens for the price of one large-model call.

![Speculative decoding: draft model proposes 5 tokens, verifier accepts 3 (teal checks) and rejects 2 (coral crosses) in one parallel forward pass](images/speculative-decoding-accept-reject.png)

In [ ]:
# ── Part 4: Speculative decoding simulation ───────────────────────────────────
def simulate_speculative_decoding(
    n_tokens_to_generate=50,
    draft_tokens_per_step=4,
    acceptance_rate=0.75,         # fraction of draft tokens accepted
    draft_latency_ms=1.0,         # small model
    verifier_latency_ms=10.0,     # large model
):
    """Simulate speculative decoding and compare to sequential generation."""

    # Sequential baseline: one verifier call per token
    t_sequential = n_tokens_to_generate * verifier_latency_ms

    # Speculative: draft K tokens, then verify once
    tokens_generated = 0
    t_speculative    = 0
    n_verifier_calls = 0

    while tokens_generated < n_tokens_to_generate:
        # Draft K tokens
        t_speculative += draft_tokens_per_step * draft_latency_ms
        # Verifier forward pass (parallel)
        t_speculative += verifier_latency_ms
        n_verifier_calls += 1
        # Accept according to acceptance rate
        accepted = int(draft_tokens_per_step * acceptance_rate) + 1  # +1 for verifier's own
        tokens_generated += min(accepted, n_tokens_to_generate - tokens_generated)

    speedup = t_sequential / t_speculative
    return t_sequential, t_speculative, speedup, n_verifier_calls

t_seq, t_spec, speedup, n_calls = simulate_speculative_decoding()
print(f"Speculative decoding simulation ({50} tokens, draft_len=4, accept_rate=75%):")
print(f"  Sequential (7B model):     {t_seq:.0f}ms  →  {50/t_seq*1000:.0f} tok/s")
print(f"  Speculative (7B+70M):      {t_spec:.0f}ms  →  {50/t_spec*1000:.0f} tok/s")
print(f"  Speedup: {speedup:.1f}×  ({n_calls} verifier calls instead of 50)")
print()
print("Prediction check:")
if speedup >= 2.0:
    print(f"  Answer (a) confirmed: {speedup:.1f}× speedup with 3-4 tokens per verifier call")
else:
    print(f"  Answer (c) — small speedup at this acceptance rate; optimal at rate > 80%")
print()

# Sweep acceptance rate
rates = np.linspace(0.5, 1.0, 11)
speedups = [simulate_speculative_decoding(acceptance_rate=r)[2] for r in rates]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rates * 100, speedups, 'o-', color='steelblue', lw=2)
ax.axhline(1.0, color='gray', ls='--', lw=1, label='No speedup baseline')
ax.set_xlabel('Token acceptance rate (%)'); ax.set_ylabel('Speedup (×)')
ax.set_title('Speculative decoding speedup vs. draft model acceptance rate')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("→ Acceptance rate > 70% required for meaningful speedup.")
print("  Small models fine-tuned on the target domain achieve 80-90% acceptance.")

---

## 🧪 Your Turn — Speculative Decoding Draft Length

**Prediction:** If we increase draft length from 4 to 8 tokens per step (more speculation), will the mean acceptance rate:

1. Go up — more speculation = more aggressive = higher throughput
2. Go down — longer chains are harder to accept wholesale
3. Stay the same — acceptance rate doesn't depend on draft length

Change `draft_tokens_per_step` to 8 in the cell below and observe.

In [ ]:
# ── 🧪 Your Turn: Change draft length ────────────────────────────────────────
# 👉 CHANGE: try draft_tokens_per_step = 8, 12 — does speedup improve?
draft_len = 4  # ← CHANGE ME

for rate in [0.6, 0.75, 0.9]:
    _, _, speedup, _ = simulate_speculative_decoding(
        draft_tokens_per_step=draft_len, acceptance_rate=rate)
    print(f"  draft_len={draft_len}, accept_rate={rate:.0%}: {speedup:.1f}× speedup")

print()
print("Observation: longer draft chains increase throughput when acceptance is high,")
print("but add overhead when acceptance is low — the optimal draft length depends on")
print("the domain-specific acceptance rate of the draft model.")

---

## Part 5 — Prefill vs. Decode: Different Phases, Different Optimal Batch Sizes

> **Riverside at 100× load:** Prefill (processing the prompt) and decode (generating tokens one by one) have very different GPU profiles. At high traffic, separating these onto different hardware can double throughput.

**Prefill:** process all prompt tokens in one parallel forward pass. High arithmetic intensity (large matmuls). Compute-bound. Benefits from large batch size.

**Decode:** generate one token at a time. Low arithmetic intensity (1-token inputs × full KV cache). Memory-bound. Benefits from large batch size differently (amortize weight reads).

This asymmetry is why production servers disaggregate prefill and decode onto separate GPU instances (disaggregated serving).


In [ ]:
# ── Part 5: Prefill vs. decode phase comparison ───────────────────────────────
PROMPT_LENGTHS = [32, 64, 128, 256]

prefill_times_ms = []
decode_times_ms  = []

model.eval()
with torch.no_grad():
    for plen in PROMPT_LENGTHS:
        # Prefill: process all tokens at once
        prompt_ids = torch.randint(0, VOCAB_SIZE, (1, plen)).to(DEVICE)
        if HAS_GPU: torch.cuda.synchronize()
        times_p = []
        for _ in range(10):
            if HAS_GPU: torch.cuda.synchronize()
            t0 = time.perf_counter()
            _, past_kvs = model(prompt_ids)
            if HAS_GPU: torch.cuda.synchronize()
            times_p.append((time.perf_counter() - t0) * 1000)
        prefill_times_ms.append(np.median(times_p))

        # Decode: one token at a time with KV cache
        times_d = []
        for _ in range(10):
            if HAS_GPU: torch.cuda.synchronize()
            t0 = time.perf_counter()
            next_id = torch.randint(0, VOCAB_SIZE, (1, 1)).to(DEVICE)
            _, _ = model(next_id, past_kvs=past_kvs)
            if HAS_GPU: torch.cuda.synchronize()
            times_d.append((time.perf_counter() - t0) * 1000)
        decode_times_ms.append(np.median(times_d))

print(f"Prefill vs. decode timing (toy model):")
print(f"{'Prompt len':12s}  {'Prefill (ms)':14s}  {'Decode/tok (ms)':16s}  {'Prefill/Decode':14s}")
print("-" * 60)
for plen, tp, td in zip(PROMPT_LENGTHS, prefill_times_ms, decode_times_ms):
    print(f"  {plen:10d}   {tp:10.2f}      {td:12.2f}           {tp/td:8.1f}×")

print()
print("Key insight: prefill (prompt processing) takes proportionally more time than decode")
print("because it processes S tokens in parallel — more FLOP per step.")
print("Decode is faster per token but limited by memory bandwidth (KV cache reads).")
print()
print("→ Production systems often use separate GPU pools: prefill (compute-intensive)")
print("  and decode (memory-intensive) with different hardware and batching strategies.")

---

## Part 6 — Toy → Real: vLLM Metrics

Each technique from Parts 1–5 maps to a metric in production serving:

| Technique | Metric improved | vLLM metric |
|---|---|---|
| KV cache | Per-token latency | TPOT (time per output token) |
| Continuous batching | Throughput | Requests/second |
| PagedAttention | Concurrent users | GPU memory utilization % |
| Speculative decoding | User-perceived speed | TTFT (time to first token) + TPOT |
| Prefill/decode split | Both | Separate TTFT and TPOT optimization |

In [ ]:
# ── Closing Decision ──────────────────────────────────────────────────────────
# Compute cumulative throughput improvement
baseline_rps = 10.0  # requests/min
kv_mult      = speedup          # from Part 1
cb_mult      = improvement      # from Part 2
sd_mult      = simulate_speculative_decoding()[2]  # from Part 4

kv_rps  = baseline_rps * kv_mult
cb_rps  = kv_rps * cb_mult
sd_rps  = cb_rps * sd_mult
target  = 1000.0

print("=" * 60)
print("  CLOSING DECISION — Riverside 100× Traffic Scaling")
print("=" * 60)
print()
print(f"  Starting point: {baseline_rps:.0f} req/min (naive serving)")
print()
print(f"  + KV cache:            {kv_rps:.0f} req/min  ({kv_mult:.1f}× from Part 1)")
print(f"  + Continuous batching: {cb_rps:.0f} req/min  ({cb_mult:.1f}× from Part 2)")
print(f"  + Speculative decoding:{sd_rps:.0f} req/min  ({sd_mult:.1f}× from Part 4)")
print()
print(f"  Target: {target:.0f} req/min")
achieved = sd_rps >= target
print(f"  Achieved with software alone: {'✓ YES' if achieved else '✗ Not quite'}")
print()
print("  Production recommendation:")
print("  1. Deploy via vLLM (KV cache + continuous batching + PagedAttention out-of-box)")
print("  2. Enable speculative decoding with a domain-fine-tuned 70M draft model")
print("  3. Monitor TTFT (user-perceived latency) and TPOT (throughput) separately")
print()
print("  One-line vLLM deployment:")
print("    vllm serve <model-name> --enable-chunked-prefill --speculative-model <draft>")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated
- KV cache — real PyTorch forward pass; verified outputs match non-cached; speedup measured
- Continuous batching — simulated; throughput comparison vs. static batching
- PagedAttention — memory utilization modeled; fragmentation vs. paged compared
- Speculative decoding — simulated; acceptance rate sweep; speedup quantified
- Prefill vs. decode — timing comparison across prompt lengths

### Tier 2 — Explained but Not Fully Built
- **PagedAttention** — the page table mechanism is modeled numerically; a real block-level attention implementation requires modifying the attention kernel

### Tier 3 — Named but Out of Scope
- **Tensor-parallel inference** — split the model across GPUs for low-latency high-throughput serving; covered in Ch5 (Distributed)
- **Disaggregated prefill-decode** — separate GPU pools for each phase; used at scale in production clusters
- **Chunked prefill** — interleave long-prompt prefill with decode to reduce TTFT jitter

---

## When to Use What

| Optimization | Benefit | When to add |
|---|---|---|
| KV cache | 3–10× faster decode | Always — built into all frameworks |
| Continuous batching | 2–3× throughput | Any serving scenario with variable lengths |
| PagedAttention | ~3× more concurrent users | When GPU memory is the constraint |
| Speculative decoding | 1.5–3× faster generation | When you have a good draft model (fine-tuned) |
| Disaggregated serving | Optimise TTFT separately | At >100 req/s with mixed short/long prompts |

→ **Next:** `learning/ai-infrastructure/08-triton-kernels/` — all the optimizations above depend on fast GPU kernels underneath. This chapter shows how to write them in Python using Triton.